# Disposal of grievances, July 2024 to June 2025

A short note in two parts. Each part asks the same four questions:

1. How many complaints were filed?
2. What share was disposed?
3. How long did a disposed complaint take, and in which stage?
4. Which three routes carried the most complaints, and where did their time go?

**Part 1 is statewide**, every department. **Part 2 is one subcategory**, set in the parameter cell below. It starts as SSEPD old age and disability pensions (OAP / ODP / MBPY / WP).

### Reading this note

**Two bases, never mixed.** Counts and the disposed share use every complaint filed in 2024-25 (**F0**). Every time figure uses **P4**: disposed complaints whose route is fully recognised and whose stage spans add up exactly to the time taken. Each part opens with a funnel from F0 to P4, and every exhibit names the step it uses.

**Repeat filings are not collapsed.** The duplicate index exists per department, not statewide. So both parts count filings, the same way.

**Three stages.** Total time to close is split into five spans that add up exactly. This note folds them into three:

| Stage | Spans | Runs from, to |
|---|---|---|
| **Processing** | registration, first assignment, closure | filing to the first hand-off, plus last action to formal closure |
| **Field action** | field action | first hand-off to being sent back up |
| **Review** | review | sent back up to the last action |

Review is zero for a complaint never sent back up. That is a real value, so the base does not change.

**Means, not medians, are split.** The mean of a total is the sum of the means of its parts. Medians do not add. Medians appear beside the means, but only the means are split.

Years run July to June. No office codes, officer names or citizen text appear.

In [ ]:
# Which subcategory Part 2 follows. Change these two lines to run it for another.
SUB_LABEL = "SSEPD pensions (OAP / ODP / MBPY / WP)"
SUB_WHERE = "dept_id = 40 AND subcategory = 'OAP/ODP/MBPY/WP'"


In [ ]:
import html

import polars as pl
from IPython.display import Markdown, display

from janasunani.analytics.bottlenecks import check_note, open_lake
from janasunani.analytics.exhibits import (
    draw_outcomes_pie, draw_stage_bar, draw_stage_stack, durations, outcomes,
    route_stage_roles, stage_means, stage_rows, stage_table, top_route_groups,
)
from janasunani.analytics.figures import Report
from janasunani.analytics.journey import STAGES_THREE
from janasunani.analytics.sample import Sample

con = open_lake()
# The progress widget is noise in an exported report.
con.execute("SET enable_progress_bar = false")
# Route names run past open_lake's 64-character display limit.
pl.Config.set_fmt_str_lengths(120)


# Tables are for a reader, not a debugger: no shape line, no dtype row, no
# quotes around text, blanks for nulls, thousands separators on counts. This
# also applies to tables the exhibits display, and survives export to Word.
def _cell(v):
    if v is None:
        return ""
    return f"{v:,}" if isinstance(v, int) else html.escape(str(v))


def _table_html(df):
    head = "".join(f"<th>{html.escape(c)}</th>" for c in df.columns)
    body = "".join("<tr>" + "".join(f"<td>{_cell(v)}</td>" for v in row) + "</tr>"
                   for row in df.rows())
    return f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"


def _table_md(df):
    rows = [list(df.columns), ["---"] * df.width,
            *[[_cell(v) for v in row] for row in df.rows()]]
    return "\n".join("| " + " | ".join(map(str, r)) + " |" for r in rows)


_fmt = get_ipython().display_formatter.formatters
_fmt["text/html"].for_type(pl.DataFrame, _table_html)
# The Word version is built from a Markdown export, which reads this one.
_fmt["text/markdown"].for_type(pl.DataFrame, _table_md)
# Silent on pass: the CA&GR Analytics Note's published 2024-25 figures.
check_note(con)

STEPS = [
    ("F0", "Complaints filed in 2024-25", "TRUE"),
    ("F1", "Disposed, with a usable closing date",
     "is_disposed AND days_to_close IS NOT NULL"),
]
FIGDIR = "../outputs/figures"


def sample(name, where):
    s = Sample(con, ns=f"{name}_", base=f"year = '2024-25' AND {where}", steps=STEPS)
    s.install_route("F1", "F1")
    s.install_phases("F1", "F1")
    return s


def show_funnel(s):
    def mean(t, w):
        return s.q(f"SELECT ROUND(AVG(days_to_close), 1) AS m FROM {t} WHERE {w}")["m"][0]

    display(s.funnel_table(["F0", "F1", "P3", "P4"]))
    f1, p4 = mean(s.t("g"), s.WHERE["F1"]), mean(s.t("phases_clean"), "TRUE")
    display(Markdown(
        f"P4 keeps {100 * s.FUNNEL['P4'] / s.FUNNEL['F1']:.0f}% of disposed complaints. "
        f"Their mean time to close is {p4} days, against {f1} for all disposed (F1). "
        "The ones left out have an office the rung ladder cannot name, or a last "
        "action dated after closure."))


def show_outcomes(s, rep, label):
    display(Markdown(f"**{s.FUNNEL['F0']:,} complaints** were filed ({label})."))
    draw_outcomes_pie(rep, outcomes(s, "F0"), f"How complaints filed in 2024-25 ended, {label}",
                      s.note("F0"), "01_outcomes")


def show_time(s, rep, label):
    means, total = stage_means(s, STAGES_THREE)
    display(durations(s).rename({"n": "Disposed", "median_days": "Median days",
                                 "mean_days": "Mean days", "slowest_tenth": "Slowest tenth"}))
    draw_stage_bar(rep, means, total, f"Where the time goes, {label}",
                   f"P4 · {s.FUNNEL['P4']:,} disposed complaints. Means, so the "
                   f"three stages add to the {total:.0f}-day average.",
                   "02_stages", STAGES_THREE)
    display(stage_table(s, total, STAGES_THREE))


def show_routes(s, rep, label, top=3, floor=100):
    groups = top_route_groups(s, top, floor)
    rows = stage_rows(s, groups, STAGES_THREE, floor)
    split = route_stage_roles(s, groups, STAGES_THREE)
    covered = sum(r["n"] for _, r in rows)
    display(Markdown("**Which offices held the complaint in each stage**, for each route. "
                     "The last column is the share of the route's complaints that split this way."))
    display(split)
    braces = [{k: row[k] for k in STAGES_THREE} for row in split.iter_rows(named=True)]
    draw_stage_stack(rep, rows, "Route", f"The {len(rows)} busiest routes, {label}",
                     f"P4 · {covered:,} of {s.FUNNEL['P4']:,} disposed complaints "
                     f"({100 * covered / s.FUNNEL['P4']:.0f}%). Means, so each bar adds "
                     "to that route's average.", "03_routes", STAGES_THREE,
                     share_of=s.FUNNEL["P4"], braces=braces)


# Part 1: statewide

Every complaint filed in 2024-25, in every department. Discarded complaints are included: they are closed before reaching any department, so they appear only here.

In [ ]:
state = sample("state", "TRUE")
rep_state = Report(con, [("Statewide", "TRUE")], FIGDIR, prefix="disp_state_")
# The statewide count is the published one, so this note ties to the CA&GR note.
assert state.FUNNEL["F0"] == 688_301, state.FUNNEL["F0"]
show_funnel(state)

## 1.1 How many complaints, and how many were disposed

In [ ]:
show_outcomes(state, rep_state, "statewide")

"Disposed" means the file was closed. "With benefit" is the part where the record says something reached the citizen. Neither says whether the decision was right.

## 1.2 How long a disposed complaint takes, by stage

In [ ]:
show_time(state, rep_state, "statewide")

## 1.3 The three busiest routes

A route is the sequence of offices that held the complaint, named by role (Collector, BDO, and so on), never by place.

**The route is set during Processing.** The office that first receives the complaint chooses where to send it. That first hand-off ends Processing and starts Field action, which covers the offices the complaint was sent down to. Review starts when the complaint first comes back up above the deepest office it reached, and covers the offices it passes on the way up.

Each bar is split into the same three stages and adds up to that route's own mean. The braces under each bar name the offices that held the complaint in that stage. Processing also includes the days from the last action to formal closure, which no office holds, so its brace names only the first office.

In [ ]:
show_routes(state, rep_state, "statewide")

# Part 2: one subcategory

The same four questions for the subcategory set at the top. Complaint types differ in how much work they need, so holding the type fixed makes the time figures easier to compare.

Discarded complaints carry no department, so the subcategory shows almost none. Its disposed share is therefore not directly comparable with the statewide one.

In [ ]:
sub = sample("sub", SUB_WHERE)
rep_sub = Report(con, [(SUB_LABEL, "TRUE")], FIGDIR, prefix="disp_sub_")
show_funnel(sub)

## 2.1 How many complaints, and how many were disposed

In [ ]:
show_outcomes(sub, rep_sub, SUB_LABEL)

## 2.2 How long a disposed complaint takes, by stage

In [ ]:
show_time(sub, rep_sub, SUB_LABEL)

## 2.3 The three busiest routes

Read as in 1.3: the route is set during Processing, and the braces name the offices in each stage.

In [ ]:
show_routes(sub, rep_sub, SUB_LABEL)

## What these numbers cannot say

- **Open complaints are not timed.** Times cover disposed complaints only, so a recent period looks faster than it will turn out.
- **P4 is a subset.** It drops complaints with an unrecognised office, and those whose last action is dated after closure. The funnel shows how many, and how the mean shifts.
- **A long stage is not idle time.** It may be field enquiry, a statutory wait, or waiting on the citizen.
- **Repeat filings count separately.** A citizen who filed three times counts three times.

In [ ]:
con.close()